# Stage 2 — Data Collection

This stage loads the raw WDI file and checks the basic structure. No feature selection, model training, or train/test split is done here.

## 2.1 Import libraries

This cell only imports packages. Configuration, folder creation, and data loading are separated below so that the workflow can be audited step by step.

In [275]:
from pathlib import Path
import re
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import pycountry
except ImportError:
    pycountry = None

import sklearn
from sklearn.base import clone
from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

try:
    import xgboost as xgb
    from xgboost import XGBRegressor
except ImportError:
    xgb = None
    XGBRegressor = None

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error

    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings("ignore", category=FutureWarning)

## 2.2 Project configuration

The target remains U5MR in 2023. Predictors use explicit pre-target years, especially 2022 values and 2018-to-2022 changes.

In [277]:
RANDOM_STATE = 42
RAW_PATH = Path("WB_WDI_WIDEF (1).csv")

TARGET_YEAR = 2023
PREDICTOR_START_YEAR = 2000
PREDICTOR_LEVEL_YEAR = 2022
MEAN_START_YEAR = 2018
MEAN_END_YEAR = 2022
CHANGE_START_YEAR = 2018
CHANGE_END_YEAR = 2022

PREDICTOR_END_YEAR = TARGET_YEAR - 1
PREDICTOR_YEARS = list(range(PREDICTOR_START_YEAR, TARGET_YEAR))
MEAN_YEARS = list(range(MEAN_START_YEAR, MEAN_END_YEAR + 1))
CHANGE_YEARS = list(range(CHANGE_START_YEAR, CHANGE_END_YEAR + 1))
MODERN_YEARS = list(range(2000, TARGET_YEAR + 1))

MAX_INDICATOR_MISSINGNESS = 0.50
MAX_FINAL_FEATURE_MISSINGNESS = 0.50
CANDIDATE_LIMIT = 60
FINAL_INDICATOR_LIMIT = 25
FINAL_ENGINEERED_FEATURE_LIMIT = 25
REDUNDANCY_LIMIT = 0.90
ENGINEERED_REDUNDANCY_LIMIT = 0.90

assert PREDICTOR_LEVEL_YEAR < TARGET_YEAR, "Predictor year must be before target year."
assert MEAN_END_YEAR < TARGET_YEAR, "Mean window must end before target year."
assert CHANGE_END_YEAR < TARGET_YEAR, "Change window must end before target year."

print("Target year:", TARGET_YEAR)
print("Predictor level year:", PREDICTOR_LEVEL_YEAR)
print("Mean feature window:", f"{MEAN_START_YEAR} to {MEAN_END_YEAR}")
print("Change feature window:", f"{CHANGE_START_YEAR} to {CHANGE_END_YEAR}")
print("Maximum original WDI indicators:", FINAL_INDICATOR_LIMIT)
print("Maximum engineered predictor columns:", FINAL_ENGINEERED_FEATURE_LIMIT)
print("Maximum missingness allowed:", MAX_INDICATOR_MISSINGNESS)

Target year: 2023
Predictor level year: 2022
Mean feature window: 2018 to 2022
Change feature window: 2018 to 2022
Maximum original WDI indicators: 25
Maximum engineered predictor columns: 25
Maximum missingness allowed: 0.5


## 2.3 Create output folders

All generated datasets, tables, plots, models, and report-ready text are saved inside one `results/` folder.

In [279]:
RESULTS_DIR = Path("results")
DATA_DIR = RESULTS_DIR / "data"
TABLE_DIR = RESULTS_DIR / "tables"
VIS_DIR = RESULTS_DIR / "visualisations"
MODEL_DIR = RESULTS_DIR / "models"
REPORT_DIR = RESULTS_DIR / "report"

for folder in [DATA_DIR, TABLE_DIR, VIS_DIR, MODEL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Output folders are ready:")
for folder in [DATA_DIR, TABLE_DIR, VIS_DIR, MODEL_DIR, REPORT_DIR]:
    print("-", folder)

Output folders are ready:
- results\data
- results\tables
- results\visualisations
- results\models
- results\report


## 2.4 Library versions

Reporting software versions improves reproducibility.

In [281]:
version_rows = [
    {"library": "Python", "version": sys.version.split()[0]},
    {"library": "pandas", "version": pd.__version__},
    {"library": "numpy", "version": np.__version__},
    {"library": "matplotlib", "version": plt.matplotlib.__version__},
    {"library": "scikit-learn", "version": sklearn.__version__}
]

if xgb is not None:
    version_rows.append({"library": "xgboost", "version": xgb.__version__})
else:
    version_rows.append({"library": "xgboost", "version": "not installed; XGBoost model skipped"})

library_versions = pd.DataFrame(version_rows)
display(library_versions)
library_versions.to_csv(TABLE_DIR / "library_versions.csv", index=False)

,library,version
0,Python,3.12.7
1,pandas,2.2.3
2,numpy,1.26.4
3,matplotlib,3.9.2
4,scikit-learn,1.5.1
5,xgboost,3.0.2


## 2.5 Load the raw WDI CSV

The raw file is loaded without feature selection or cleaning. Cleaning starts in Stage 3.

In [283]:
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {RAW_PATH}. Put the raw CSV in the same folder as this notebook."
    )

raw = pd.read_csv(RAW_PATH, low_memory=False)

print("Raw dataset shape:", raw.shape)
display(raw.head(3))

Raw dataset shape: (214891, 107)


,STRUCTURE,STRUCTURE_ID,ACTION,FREQ,REF_AREA,INDICATOR,SEX,AGE,URBANISATION,UNIT_MEASURE,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,MNE,WB_WDI_IS_RRS_TOTL_KM,_T,_T,_T,KM,...,2.490000e+02,2.490000e+02,2.490000e+02,2.500000e+02,NaN,NaN,NaN,NaN,NaN,NaN
1,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,MMR,WB_WDI_ST_INT_XPND_CD,_T,_T,_T,USD,...,2.010000e+08,1.360000e+08,1.180000e+08,2.140000e+08,NaN,NaN,NaN,NaN,NaN,NaN
2,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,LUX,WB_WDI_SL_FAM_WORK_FE_ZS,F,_T,_T,PT,...,1.394918e+00,9.868241e-01,8.426004e-01,1.478158e+00,1.6546,1.686016,2.180262,1.688659,1.988782,1.98045


## 2.6 Raw structure audit

This audit documents the raw file structure before any transformation: number of rows, columns, year columns, entities, and indicators.

In [285]:
year_columns_raw = sorted(
    [c for c in raw.columns if re.fullmatch(r"\d{4}", str(c))],
    key=int
)

raw_overview = pd.DataFrame({
    "item": [
        "rows",
        "columns",
        "year columns",
        "first year",
        "last year",
        "unique entities",
        "unique indicators"
    ],
    "value": [
        raw.shape[0],
        raw.shape[1],
        len(year_columns_raw),
        min(map(int, year_columns_raw)) if year_columns_raw else np.nan,
        max(map(int, year_columns_raw)) if year_columns_raw else np.nan,
        raw["REF_AREA"].nunique() if "REF_AREA" in raw.columns else np.nan,
        raw["INDICATOR"].nunique() if "INDICATOR" in raw.columns else np.nan
    ]
})

display(raw_overview)
raw_overview.to_csv(TABLE_DIR / "raw_data_overview.csv", index=False)

,item,value
0,rows,214891
1,columns,107
2,year columns,66
3,first year,1960
4,last year,2025
5,unique entities,265
6,unique indicators,1516


## 2.7 Raw data types

The raw WDI file contains metadata columns and year columns. Year columns are converted to numeric values in Stage 3, not here, so this cell only audits the original types.

In [287]:
raw_dtype_summary = (
    raw.dtypes.astype(str)
    .value_counts()
    .rename_axis("dtype")
    .reset_index(name="number_of_columns")
)

raw_column_audit = pd.DataFrame({
    "column": raw.columns,
    "dtype_before_cleaning": raw.dtypes.astype(str).values,
    "non_null_count": raw.notna().sum().values,
    "missing_count": raw.isna().sum().values,
    "missing_pct": raw.isna().mean().values
}).sort_values(["missing_pct", "column"], ascending=[False, True])

year_dtype_audit = raw_column_audit[raw_column_audit["column"].astype(str).isin(year_columns_raw)].copy()
metadata_dtype_audit = raw_column_audit[~raw_column_audit["column"].astype(str).isin(year_columns_raw)].copy()

print("Raw dtype summary:")
display(raw_dtype_summary)

print("Metadata column dtypes:")
display(metadata_dtype_audit)

print("Year-column dtype examples before numeric conversion:")
display(year_dtype_audit.head(10))

raw_column_audit.to_csv(TABLE_DIR / "raw_column_dtype_and_missingness_audit.csv", index=False)

Raw dtype summary:


,dtype,number_of_columns
0,float64,66
1,object,39
2,int64,2


Metadata column dtypes:


,column,dtype_before_cleaning,non_null_count,missing_count,missing_pct
2,ACTION,object,214891,0,0.0
7,AGE,object,214891,0,0.0
27,AGE_LABEL,object,214891,0,0.0
13,AGG_METHOD,object,214891,0,0.0
33,AGG_METHOD_LABEL,object,214891,0,0.0
18,COMMENT_TS,object,214891,0,0.0
10,COMP_BREAKDOWN_1,object,214891,0,0.0
30,COMP_BREAKDOWN_1_LABEL,object,214891,0,0.0
11,COMP_BREAKDOWN_2,object,214891,0,0.0
31,COMP_BREAKDOWN_2_LABEL,object,214891,0,0.0


Year-column dtype examples before numeric conversion:


,column,dtype_before_cleaning,non_null_count,missing_count,missing_pct
106,2025,float64,11626,203265,0.945898
41,1960,float64,27583,187308,0.871642
42,1961,float64,31557,183334,0.853149
43,1962,float64,32509,182382,0.848719
44,1963,float64,33198,181693,0.845512
45,1964,float64,33567,181324,0.843795
46,1965,float64,34884,180007,0.837667
47,1966,float64,35104,179787,0.836643
48,1967,float64,35598,179293,0.834344
49,1968,float64,36033,178858,0.832320


## 2.8 Raw missingness audit

This gives an initial view of missingness before country filtering and before choosing the modern 2000–2023 period.

In [289]:
year_missingness_raw = pd.DataFrame({
    "year": [int(c) for c in year_columns_raw],
    "available_values": [raw[c].notna().sum() for c in year_columns_raw],
    "missing_values": [raw[c].isna().sum() for c in year_columns_raw],
    "missing_pct": [raw[c].isna().mean() for c in year_columns_raw]
})

print("Most recent year missingness:")
display(year_missingness_raw.tail(10))

print("Columns with highest missingness:")
display(raw_column_audit.head(15))

year_missingness_raw.to_csv(TABLE_DIR / "raw_year_missingness.csv", index=False)

Most recent year missingness:


,year,available_values,missing_values,missing_pct
56,2016,168776,46115,0.214597
57,2017,167226,47665,0.221810
58,2018,166464,48427,0.225356
59,2019,164066,50825,0.236515
60,2020,161684,53207,0.247600
61,2021,157759,57132,0.265865
62,2022,147454,67437,0.313820
63,2023,133514,81377,0.378690
64,2024,91019,123872,0.576441
65,2025,11626,203265,0.945898


Columns with highest missingness:


,column,dtype_before_cleaning,non_null_count,missing_count,missing_pct
106,2025,float64,11626,203265,0.945898
41,1960,float64,27583,187308,0.871642
42,1961,float64,31557,183334,0.853149
43,1962,float64,32509,182382,0.848719
44,1963,float64,33198,181693,0.845512
45,1964,float64,33567,181324,0.843795
46,1965,float64,34884,180007,0.837667
47,1966,float64,35104,179787,0.836643
48,1967,float64,35598,179293,0.834344
49,1968,float64,36033,178858,0.832320


## 2.9 Raw duplicate audit

Duplicate checks are documented before cleaning. The country-indicator duplicate check is especially important because the final feature construction assumes one row per country-indicator pair.

In [291]:
full_duplicate_rows = int(raw.duplicated().sum())

duplicate_audit = {
    "full_duplicate_rows": full_duplicate_rows
}

if {"REF_AREA", "INDICATOR"}.issubset(raw.columns):
    duplicate_key_mask = raw.duplicated(subset=["REF_AREA", "INDICATOR"], keep=False)
    duplicate_audit["duplicate_ref_area_indicator_rows"] = int(duplicate_key_mask.sum())
    duplicate_audit["duplicate_ref_area_indicator_pairs"] = int(
        raw.loc[duplicate_key_mask, ["REF_AREA", "INDICATOR"]].drop_duplicates().shape[0]
    )

    duplicate_key_examples = (
        raw.loc[duplicate_key_mask, ["REF_AREA", "REF_AREA_LABEL", "INDICATOR", "INDICATOR_LABEL"]]
        .drop_duplicates()
        .head(20)
    )
else:
    duplicate_key_examples = pd.DataFrame()
    duplicate_audit["duplicate_ref_area_indicator_rows"] = np.nan
    duplicate_audit["duplicate_ref_area_indicator_pairs"] = np.nan

duplicate_audit_table = pd.DataFrame(
    [{"check": key, "value": value} for key, value in duplicate_audit.items()]
)

display(duplicate_audit_table)
display(duplicate_key_examples)

duplicate_audit_table.to_csv(TABLE_DIR / "raw_duplicate_audit.csv", index=False)
duplicate_key_examples.to_csv(TABLE_DIR / "raw_duplicate_key_examples.csv", index=False)

,check,value
0,full_duplicate_rows,0
1,duplicate_ref_area_indicator_rows,0
2,duplicate_ref_area_indicator_pairs,0


,REF_AREA,REF_AREA_LABEL,INDICATOR,INDICATOR_LABEL
